# Setup

---



In [ ]:
# Run once per environment if entsoe-py is not installed
%pip install -q entsoe-py

In [ ]:
# Standard library
import json
import os
import re
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Optional

# Third-party libraries
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# scikit-learn
from sklearn.linear_model import LassoCV, LassoLarsCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

# statsmodels
from scipy.stats import norm

# ENTSO-E
from entsoe import EntsoePandasClient, EntsoeRawClient
from entsoe.parsers import parse_prices

# Environment
from dotenv import load_dotenv
load_dotenv()

# Settings
---

In [ ]:
# -----------------------------------------------------------------------------
# Repository root (set automatically relative to this notebook)
# -----------------------------------------------------------------------------
REPO_ROOT = Path.cwd().parent.parent

# -----------------------------------------------------------------------------
# Global configuration
# -----------------------------------------------------------------------------
TARGET_TZ = "Europe/Berlin"
POST_REGIME_START = pd.Timestamp("2025-10-01 00:00", tz=TARGET_TZ)

# -----------------------------------------------------------------------------
# ENTSO-E API
# -----------------------------------------------------------------------------
COUNTRY_CODE_ENTSOE = "DE_LU"
API_KEY = os.getenv("ENTSOE_API_KEY")

if API_KEY is None:
    raise ValueError("Environment variable 'ENTSOE_API_KEY' is not set.")

ENTSOE_START_DATE = pd.Timestamp("2024-01-01", tz=TARGET_TZ)
ENTSOE_END_DATE = pd.Timestamp("2026-03-10", tz=TARGET_TZ)

# -----------------------------------------------------------------------------
# Local data directories
# -----------------------------------------------------------------------------
# Set these paths to your local ERA5 and ICON data directories.
# ERA5 CSVs are expected as one file per day, grouped by year.
# ICON CSVs are expected in a single directory with one file per forecast run.
ERA5_2024_DIR = Path("data/era5/2024")
ERA5_2025_DIR = Path("data/era5/2025")
ERA5_2026_DIR = Path("data/era5/2026")

ICON_DIR      = Path("data/icon")
DWD_FOLDER_OFFSET_DATE = date(2025, 10, 26)
START_FOLDER_DATE = date(2025, 8, 1)
REQUIRED_RUN = "09"

SKIP_DATES = {
    date(2025, 10, 26),
    date(2025, 10, 27),
    date(2025, 10, 28),
}

# -----------------------------------------------------------------------------
# LEAR operational model
# -----------------------------------------------------------------------------
VST_BOOLEAN = True
WEATHER_SOURCE_OP = "DWD"  # "ERA5" or "DWD"
USE_EXAA_OP = False
USE_EXAA_ONLY_OP = False
LARS_START_DATE_OPERATIONAL = pd.Timestamp("2025-12-01", tz=TARGET_TZ)

TEST_START_P = pd.Timestamp("2025-12-01", tz=TARGET_TZ)
TEST_END_P = pd.Timestamp("2025-12-02 23:45", tz=TARGET_TZ)
TRAIN_DAYS_ROLLING_P = 364
N_CLUSTERS_OP = 5

EXPERIMENT_NAME_OP = (
    f"lear_{WEATHER_SOURCE_OP.lower() if not USE_EXAA_ONLY_OP else 'exaa_only'}"
    + (f"_exaa" if USE_EXAA_OP and not USE_EXAA_ONLY_OP else "_fundamental" if not USE_EXAA_ONLY_OP else "")
    + (f"_c{N_CLUSTERS_OP}" if not USE_EXAA_ONLY_OP else "")
    + f"_d{TRAIN_DAYS_ROLLING_P}"
)

if USE_EXAA_ONLY_OP:
    EXPORT_BASE_P = REPO_ROOT / "results" / "lear_op_results" / "exaa_only" / f"d{TRAIN_DAYS_ROLLING_P}"
elif USE_EXAA_OP:
    EXPORT_BASE_P = REPO_ROOT / "results" / "lear_op_results" / WEATHER_SOURCE_OP.lower() / f"d{TRAIN_DAYS_ROLLING_P}" / f"c{N_CLUSTERS_OP}" / "exaa"
else:
    EXPORT_BASE_P = REPO_ROOT / "results" / "lear_op_results" / WEATHER_SOURCE_OP.lower() / f"d{TRAIN_DAYS_ROLLING_P}" / f"c{N_CLUSTERS_OP}" / "fundamental"

# -----------------------------------------------------------------------------
# LEAR ANC analysis
# -----------------------------------------------------------------------------
WEATHER_SOURCE_ANC = "ERA5"  # "ERA5" or "DWD"
USE_EXAA_ANC = True
USE_EXAA_ONLY_ANC = False
LARS_START_DATE_ANC = pd.Timestamp("2025-12-01", tz=TARGET_TZ)

TEST_START_ANC = pd.Timestamp("2025-12-01", tz=TARGET_TZ)
TEST_END_ANC = pd.Timestamp("2026-02-28 23:45", tz=TARGET_TZ)
TRAIN_DAYS_ROLLING_ANC = 112
N_CLUSTERS_ANC = 5

ANC_MTU_WINDOW_WIND = range(0, 96)
ANC_MTU_WINDOW_SOLAR = range(0, 96)

EXPERIMENT_NAME_ANC = (
    f"anc_{WEATHER_SOURCE_ANC.lower() if not USE_EXAA_ONLY_ANC else 'exaa_only'}"
    + (f"_exaa" if USE_EXAA_ANC and not USE_EXAA_ONLY_ANC else "_fundamental" if not USE_EXAA_ONLY_ANC else "")
    + (f"_c{N_CLUSTERS_ANC}" if not USE_EXAA_ONLY_ANC else "")
    + f"_d{TRAIN_DAYS_ROLLING_ANC}"
)

if USE_EXAA_ONLY_ANC:
    EXPORT_PATH_ANC_BASE = REPO_ROOT / "results" / "lear_anc_results" / "exaa_only" / f"d{TRAIN_DAYS_ROLLING_ANC}"
elif USE_EXAA_ANC:
    EXPORT_PATH_ANC_BASE = REPO_ROOT / "results" / "lear_anc_results" / WEATHER_SOURCE_ANC.lower() / f"c{N_CLUSTERS_ANC}" / f"d{TRAIN_DAYS_ROLLING_ANC}" / "exaa"
else:
    EXPORT_PATH_ANC_BASE = REPO_ROOT / "results" / "lear_anc_results" / WEATHER_SOURCE_ANC.lower() / f"c{N_CLUSTERS_ANC}" / f"d{TRAIN_DAYS_ROLLING_ANC}" / "fundamental"

EXPORT_PATH_ANC_FEATURES = EXPORT_PATH_ANC_BASE / "anc_all_feature_results.csv"
EXPORT_PATH_ANC_WIND     = EXPORT_PATH_ANC_BASE / "anc_wind_feature_results.csv"
EXPORT_PATH_ANC_SOLAR    = EXPORT_PATH_ANC_BASE / "anc_solar_feature_results.csv"

# Data
---

## EPEX DE-LU Prices (ENTSO-E API)


This section retrieves day-ahead auction prices for the DE-LU bidding zone via the ENTSO-E API.

The raw ENTSO-E response contains quarter-hourly values in the pre-October-2025 regime (forward-filled prices) and thereafter (real 15-min prices).

In [ ]:
def fetch_prices(start_day: pd.Timestamp, end_day: pd.Timestamp) -> pd.DataFrame:
    """
    Fetch DE-LU SDAC day-ahead prices from ENTSO-E and return a
    15-minute Europe/Berlin-aligned price series.

    Parameters
    ----------
    start_day : pd.Timestamp
        First calendar day to include (timezone-aware).
    end_day : pd.Timestamp
        Last calendar day to include (timezone-aware).

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by `timestamp` with column `price_da` in EUR/MWh.

    Notes
    -----
    DST-aware daily grids imply 92 MTUs on spring transition days and
    100 MTUs on autumn transition days. These are treated as valid.
    Only genuinely incomplete days are invalidated.
    """
    client = EntsoePandasClient(api_key=API_KEY)

    query_start = start_day.tz_convert(TARGET_TZ)
    query_end = (end_day + pd.Timedelta(days=1)).tz_convert(TARGET_TZ)

    raw_series = client.query_day_ahead_prices(
        COUNTRY_CODE_ENTSOE,
        start=query_start,
        end=query_end,
    )

    if raw_series.empty:
        raise ValueError("No day-ahead price data returned by the ENTSO-E API.")

    raw_series = raw_series.tz_convert(TARGET_TZ).rename("price_da")

    # Construct full 15-minute target grid and forward-fill hourly values.
    full_index = pd.date_range(
        start=raw_series.index.min().normalize(),
        end=raw_series.index.max().normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=15),
        freq="15min",
        tz=TARGET_TZ,
    )
    series_15 = raw_series.reindex(full_index).ffill(limit=3)

    # Invalidate only days with fewer observations than implied by the DST-aware grid.
    expected_counts = pd.Series(1, index=full_index).groupby(full_index.normalize()).transform("count")
    actual_counts = series_15.groupby(series_15.index.normalize()).transform("count")
    series_15 = series_15.where(actual_counts >= expected_counts)

    # Restrict output to the requested calendar window.
    start_cut = start_day.tz_convert(TARGET_TZ).normalize()
    end_cut = end_day.tz_convert(TARGET_TZ).normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=15)

    df_prices_15 = (
        series_15.loc[start_cut:end_cut]
        .to_frame(name="price_da")
        .rename_axis("timestamp")
        .sort_index()
    )

    return df_prices_15

## EXAA DE-LU Prices (ENTSO-E API)

This section retrieves day-ahead auction prices for the DE-LU bidding zone via the ENTSO-E API.
The raw ENTSO-E response contains quarter-hourly values throughout, representing the native solution. 

In [ ]:
def fetch_prices_exaa(start_day: pd.Timestamp, end_day: pd.Timestamp) -> pd.DataFrame:
    """
    Fetch DE-LU EXAA day-ahead prices (ENTSO-E Sequence 2) and return a
    15-minute Europe/Berlin-aligned price series.

    The ENTSO-E 100-document response limit is handled via time chunking.

    Parameters
    ----------
    start_day : pd.Timestamp
        First calendar day to include (timezone-aware).
    end_day : pd.Timestamp
        Last calendar day to include (timezone-aware).

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by `timestamp` with column `price_exaa` in EUR/MWh.

    Notes
    -----
    DST-aware daily grids imply 92 MTUs on spring transition days and
    100 MTUs on autumn transition days. These are treated as valid.
    Only genuinely incomplete days are invalidated.
    """
    client = EntsoeRawClient(api_key=API_KEY)
    chunk_days = 90

    all_series = []
    current_start = start_day.normalize()

    while current_start <= end_day:
        current_end = min(current_start + pd.Timedelta(days=chunk_days - 1), end_day)

        query_start = current_start.tz_convert(TARGET_TZ)
        query_end = (current_end + pd.Timedelta(days=1)).tz_convert(TARGET_TZ)

        xml = client.query_day_ahead_prices(
            COUNTRY_CODE_ENTSOE,
            start=query_start,
            end=query_end,
            sequence=2,
        )
        parsed = parse_prices(xml)

        if isinstance(parsed, dict):
            chunk_series = next(iter(parsed.values())) if len(parsed) == 1 else max(parsed.values(), key=len)
        else:
            chunk_series = parsed

        if chunk_series is None or len(chunk_series) == 0:
            current_start = current_end + pd.Timedelta(days=1)
            continue

        if getattr(chunk_series.index, "tz", None) is None:
            chunk_series = chunk_series.tz_localize("UTC")

        chunk_series = chunk_series.tz_convert(TARGET_TZ).rename("price_exaa")
        all_series.append(chunk_series)

        current_start = current_end + pd.Timedelta(days=1)

    series = pd.concat(all_series).sort_index()
    series = series[~series.index.duplicated(keep="last")]

    # Construct full 15-minute target grid and forward-fill hourly values.
    full_index = pd.date_range(
        start=series.index.min().normalize(),
        end=series.index.max().normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=15),
        freq="15min",
        tz=TARGET_TZ,
    )
    series_15 = series.reindex(full_index).ffill(limit=3)

    # Invalidate only days with fewer observations than implied by the DST-aware grid.
    expected_counts = pd.Series(1, index=full_index).groupby(full_index.normalize()).transform("count")
    actual_counts = series_15.groupby(series_15.index.normalize()).transform("count")
    series_15 = series_15.where(actual_counts >= expected_counts)

    # Restrict output to the requested calendar window.
    start_cut = start_day.tz_convert(TARGET_TZ).normalize()
    end_cut = end_day.tz_convert(TARGET_TZ).normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=15)

    df_prices_exaa_15 = (
        series_15.loc[start_cut:end_cut]
        .to_frame(name="price_exaa")
        .rename_axis("timestamp")
        .sort_index()
    )

    return df_prices_exaa_15

## Load DE-LU (ENTSO-E API)

In [ ]:
def fetch_load_forecast(start_day: pd.Timestamp, end_day: pd.Timestamp) -> pd.DataFrame:
    """
    Fetch the ENTSO-E day-ahead load forecast for the DE-LU bidding zone
    and return a 15-minute Europe/Berlin-aligned series.

    Parameters
    ----------
    start_day : pd.Timestamp
        First calendar day to include (timezone-aware).
    end_day : pd.Timestamp
        Last calendar day to include (timezone-aware).

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by `timestamp` with column `load_fc` in MW.

    Notes
    -----
    DST-aware daily grids imply 92 MTUs on spring transition days and
    100 MTUs on autumn transition days. These are treated as valid.
    Only genuinely incomplete days are invalidated.
    """
    client = EntsoePandasClient(api_key=API_KEY)

    query_start = start_day.tz_convert(TARGET_TZ)
    query_end = (end_day + pd.Timedelta(days=1)).tz_convert(TARGET_TZ)

    obj = client.query_load_forecast(
        COUNTRY_CODE_ENTSOE,
        start=query_start,
        end=query_end,
    )

    if isinstance(obj, pd.DataFrame):
        numeric_cols = obj.select_dtypes(include="number").columns.tolist()
        if not numeric_cols:
            raise ValueError(f"No numeric columns found in load forecast response: {obj.columns.tolist()}")
        series = obj[numeric_cols[0]].copy()
    else:
        series = obj.copy()

    series = series.tz_convert(TARGET_TZ)
    series.name = "load_fc"

    # Construct full 15-minute target grid and forward-fill hourly values.
    full_index = pd.date_range(
        start=series.index.min().normalize(),
        end=series.index.max().normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=15),
        freq="15min",
        tz=TARGET_TZ,
    )
    series_15 = series.reindex(full_index).ffill(limit=3)

    # Invalidate only days with fewer observations than implied by the DST-aware grid.
    expected_counts = pd.Series(1, index=full_index).groupby(full_index.normalize()).transform("count")
    actual_counts = series_15.groupby(series_15.index.normalize()).transform("count")
    series_15 = series_15.where(actual_counts >= expected_counts)

    # Restrict output to the requested calendar window.
    start_cut = start_day.tz_convert(TARGET_TZ).normalize()
    end_cut = end_day.tz_convert(TARGET_TZ).normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=15)

    df_load_forecast_15 = (
        series_15.loc[start_cut:end_cut]
        .to_frame(name="load_fc")
        .rename_axis("timestamp")
        .sort_index()
    )

    return df_load_forecast_15

## ERA5 Weather Data (2024, 2025, 2026)

In [ ]:
def load_era5(
    dirs: list[Path],
    target_tz: str = "Europe/Berlin",
) -> pd.DataFrame:
    """
    Load clustered ERA5 CSV files from multiple yearly directories and merge
    them into a single timezone-aware DataFrame.

    Each directory is expected to contain one CSV per variable. Within each
    year, files are joined on the timestamp index. The yearly DataFrames are
    then concatenated, duplicate timestamps removed, and the index converted
    to the target timezone.

    Parameters
    ----------
    dirs : list[Path]
        Directories containing clustered ERA5 CSV files.
    target_tz : str, default "Europe/Berlin"
        Target timezone of the output index.

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by `timestamp` with columns named
        `{var}_{original_column}`.
    """
    yearly_dfs = []

    for base_dir in dirs:
        csv_files = sorted(f for f in os.listdir(base_dir) if f.endswith(".csv"))

        if not csv_files:
            raise FileNotFoundError(f"No CSV files found in directory: {base_dir}")

        variable_dfs = []

        for filename in csv_files:
            var_name = filename.split("_")[0]

            df_var = pd.read_csv(
                base_dir / filename,
                comment="#",
                parse_dates=["timestamp"],
            )
            df_var = df_var.set_index("timestamp")
            df_var = df_var.rename(columns={col: f"{var_name}_{col}" for col in df_var.columns})

            variable_dfs.append(df_var)

        df_year = variable_dfs[0].copy()
        for df_next in variable_dfs[1:]:
            df_year = df_year.join(df_next, how="inner")

        yearly_dfs.append(df_year)

    df_era5 = pd.concat(yearly_dfs).sort_index()
    df_era5 = df_era5.loc[~df_era5.index.duplicated(keep="first")]

    if df_era5.index.tz is None:
        df_era5.index = df_era5.index.tz_localize("UTC")

    df_era5.index = df_era5.index.tz_convert(target_tz)
    df_era5.index.name = "timestamp"

    return df_era5

## DWD ICON-D2 Weather Data (available from 01/08/2025 onwards)

In [ ]:
def load_dwd(
    icon_dir: Path,
    start_folder_date: date,
    required_run: str,
    skip_dates: set[date] = frozenset(),
    folder_offset_date: date = DWD_FOLDER_OFFSET_DATE,
    target_tz: str = "Europe/Berlin",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load clustered DWD ICON-D2 forecast data from the daily folder structure.

    The function iterates over all eligible daily folders in `icon_dir`,
    resolves the associated forecast date, loads all variable-specific CSVs
    for the required model run, and returns two native-resolution DataFrames:

    - `df_hourly`: hourly wind and temperature data (`t2m`, `u10`, `v10`)
    - `df_qh`: quarter-hourly solar radiation data (`ASWDIR`, `ASWDIFD`),
      shifted from interval-end to interval-start timestamps (-15 minutes)

    Folder naming convention
    ------------------------
    Before `folder_offset_date`, the folder date equals the forecast date.
    From `folder_offset_date` onwards, the forecast date is defined as
    folder date + 1 day. This reflects a change in the DWD delivery naming
    convention.

    Parameters
    ----------
    icon_dir : Path
        Root directory containing one subfolder per forecast day.
    start_folder_date : date
        First folder date to include.
    required_run : str
        ICON model run hour to load, e.g. "09".
    skip_dates : set[date], default frozenset()
        Folder dates to skip.
    folder_offset_date : date, default DWD_FOLDER_OFFSET_DATE
        First folder date for which folder date + 1 corresponds to the
        forecast date.
    target_tz : str, default "Europe/Berlin"
        Target timezone of the output indices.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        `df_hourly` contains hourly `t2m`, `u10`, and `v10` data.
        `df_qh` contains quarter-hourly `ASWDIR` and `ASWDIFD` data with
        interval-start timestamps.
    """
    if not icon_dir.exists():
        raise FileNotFoundError(f"ICON directory not found: {icon_dir}")

    hourly_dfs = []
    qh_dfs = []

    for folder_name in sorted(os.listdir(icon_dir)):
        folder_path = icon_dir / folder_name
        if not folder_path.is_dir():
            continue

        try:
            date_str = folder_name.split("_")[3]
            folder_date = datetime.strptime(date_str, "%Y%m%d").date()
        except Exception:
            continue

        if folder_date < start_folder_date or folder_date in skip_dates:
            continue

        # Resolve forecast date from the folder naming convention.
        forecast_date = folder_date if folder_date < folder_offset_date else folder_date + timedelta(days=1)

        variable_dfs = []

        for filename in sorted(os.listdir(folder_path)):
            if not filename.endswith(".csv"):
                continue

            parts = filename.replace(".csv", "").split("_")
            run_hour = next((token[-2:] for token in parts if token.isdigit() and len(token) == 10), "")

            if run_hour != required_run:
                continue

            var_name = parts[0]
            df_var = pd.read_csv(folder_path / filename, comment="#", sep=",", engine="python")

            if "timestamp" not in df_var.columns:
                raise ValueError(f"Missing 'timestamp' column in: {filename}")

            df_var = df_var.rename(columns={col: f"{var_name}_{col}" for col in df_var.columns if col.startswith("cluster_")})
            df_var["timestamp"] = pd.to_datetime(df_var["timestamp"], utc=True).dt.tz_convert(target_tz)

            variable_dfs.append(df_var)

        if not variable_dfs:
            raise ValueError(f"No CSVs with run={required_run} found in: {folder_name}")

        df_day = variable_dfs[0].copy()
        for df_next in variable_dfs[1:]:
            df_day = df_day.merge(df_next, on="timestamp", how="outer")
        df_day = df_day.sort_values("timestamp")

        # Define the local forecast-day window.
        start = pd.Timestamp(forecast_date, tz=target_tz)
        end = start + pd.Timedelta(days=1)

        hourly_cols = ["timestamp"] + [col for col in df_day.columns if col.startswith(("t2m_", "u10_", "v10_"))]
        qh_cols = ["timestamp"] + [col for col in df_day.columns if col.startswith(("ASWDIR_", "ASWDIFD_"))]

        # Keep only hourly observations on the forecast day.
        df_hourly_day = (
            df_day[hourly_cols]
            .loc[
                (df_day["timestamp"] >= start)
                & (df_day["timestamp"] < end)
                & (df_day["timestamp"].dt.minute == 0)
            ]
            .copy()
        )

        # Shift solar data from interval-end to interval-start and then
        # restrict to the local forecast day.
        df_qh_day = df_day[qh_cols].copy()
        df_qh_day["timestamp"] = df_qh_day["timestamp"] - pd.Timedelta(minutes=15)
        df_qh_day = df_qh_day.loc[
            (df_qh_day["timestamp"] >= start)
            & (df_qh_day["timestamp"] < end)
        ]

        hourly_dfs.append(df_hourly_day)
        qh_dfs.append(df_qh_day)

    if not hourly_dfs:
        raise ValueError("No hourly DWD data loaded.")
    if not qh_dfs:
        raise ValueError("No quarter-hourly DWD data loaded.")

    df_hourly = pd.concat(hourly_dfs, ignore_index=True).sort_values("timestamp").set_index("timestamp")
    df_qh = pd.concat(qh_dfs, ignore_index=True).sort_values("timestamp").set_index("timestamp")

    df_hourly.index.name = "timestamp"
    df_qh.index.name = "timestamp"

    return df_hourly, df_qh

## Data Validation (TBD)

# Feature Engineering

---





## ERA5 - Weather Features

In [ ]:
def build_era5_features(
    df_era5: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build daily vector features from hourly ERA5 data.

    For each calendar day, hourly ERA5 values are reshaped into a flat daily
    profile with columns of the form `{var}_cluster_{i}_h{hh}`. Wind speed is
    derived from the `u10` and `v10` components before pivoting.

    Parameters
    ----------
    df_era5 : pd.DataFrame
        Hourly ERA5 DataFrame with timezone-aware index and columns such as
        `ssrd_cluster_{i}`, `u10_cluster_{i}`, and `v10_cluster_{i}`.

    Returns
    -------
    pd.DataFrame
        Daily feature matrix indexed by local calendar date
        (`Europe/Berlin`, timezone-aware). Columns follow the pattern
        `ssrd_cluster_{i}_h{hh}` and `wind_speed_cluster_{i}_h{hh}`.
    """
    df = df_era5.copy()

    ts_local = df.index.tz_convert("Europe/Berlin")
    df["date"] = ts_local.date
    df["hour"] = ts_local.hour

    # Derive scalar wind speed from the u10 and v10 components.
    u_cols = [col for col in df.columns if col.startswith("u10_cluster_")]
    for u_col in u_cols:
        cluster_id = u_col.split("_")[-1]
        v_col = f"v10_cluster_{cluster_id}"

        if v_col in df.columns:
            df[f"wind_speed_cluster_{cluster_id}"] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)

    feature_cols = (
        [col for col in df.columns if col.startswith("ssrd_cluster_")]
        + [col for col in df.columns if col.startswith("wind_speed_cluster_")]
    )

    # Pivot to one row per local calendar day.
    df_features = df.pivot_table(
        index="date",
        columns="hour",
        values=feature_cols,
        aggfunc="mean",
    )
    df_features.columns = [f"{col_name}_h{int(hour):02d}" for col_name, hour in df_features.columns]

    df_features = df_features.interpolate(axis=1, limit=1, limit_area="inside")
    df_features.index = pd.to_datetime(df_features.index).tz_localize("Europe/Berlin")
    df_features.index.name = "date"

    return df_features

## DWD ICON-D2 - Weather Features

In [ ]:
def build_dwd_features(
    df_hourly: pd.DataFrame,
    df_qh: pd.DataFrame,
    tz_local: str = "Europe/Berlin",
) -> pd.DataFrame:
    """
    Build DWD ICON-D2 daily vector features (one row per calendar day).

    Wind features are derived from hourly u10/v10 components and pivoted
    into 24-hour daily profiles with columns 'wind_speed_cluster_{i}_h{hh}'.
    Solar features are taken from the native 15-minute data and pivoted into
    96-MTU daily profiles with columns 'sw_dir_cluster_{i}_mtu{mm}' and
    'sw_dif_cluster_{i}_mtu{mm}'.

    Note: df_hourly may also contain t2m columns, which are not used here.

    Parameters
    ----------
    df_hourly : pd.DataFrame
        Hourly DWD DataFrame with u10/v10 cluster columns.
    df_qh : pd.DataFrame
        15-minute DWD DataFrame with ASWDIR/ASWDIFD cluster columns.
    tz_local : str
        Local timezone string. Defaults to 'Europe/Berlin'.

    Returns
    -------
    pd.DataFrame
        Feature matrix with one row per calendar day (Midnight tz-aware
        timestamps in tz_local). Days without data are included as NaN rows.
    """
    # ------------------------------------------------------------------
    # Wind features (hourly resolution)
    # ------------------------------------------------------------------
    df_w = df_hourly.copy()
    ts_loc_w = df_w.index.tz_convert(tz_local)
    df_w = df_w.assign(
        date=ts_loc_w.date,
        hour=ts_loc_w.hour,
    )

    u_cols = [c for c in df_w.columns if c.startswith("u10_cluster_")]

    wind_data = {}
    for u_col in u_cols:
        i = u_col.split("_")[-1]
        v_col = f"v10_cluster_{i}"
        if v_col in df_w.columns:
            speed_col = f"wind_speed_cluster_{i}"
            wind_data[speed_col] = np.sqrt(df_w[u_col].to_numpy() ** 2 + df_w[v_col].to_numpy() ** 2)

    if not wind_data:
        raise ValueError("Function build_dwd_features: no matching u10/v10 cluster pairs found.")

    df_w_base = df_w[["date", "hour"]].copy()
    df_w_speed = pd.DataFrame(wind_data, index=df_w.index)
    df_w_full = pd.concat([df_w_base, df_w_speed], axis=1)

    wind_cols = list(df_w_speed.columns)

    df_wind_pivot = df_w_full.pivot_table(
        index="date",
        columns="hour",
        values=wind_cols,
        aggfunc="mean",
    )
    df_wind_pivot.columns = [
        f"{col[0]}_h{int(col[1]):02d}" for col in df_wind_pivot.columns
    ]
    df_wind_pivot = df_wind_pivot.interpolate(axis=1, limit=1, limit_area="inside")

    # ------------------------------------------------------------------
    # Solar features (15-min resolution)
    # ------------------------------------------------------------------
    df_s = df_qh.copy()
    ts_loc_s = df_s.index.tz_convert(tz_local)
    df_s = df_s.assign(
        date=ts_loc_s.date,
        mtu=ts_loc_s.hour * 4 + ts_loc_s.minute // 15,
    )

    dir_cols = [c for c in df_s.columns if c.startswith("ASWDIR_cluster_")]

    solar_data = {}
    for dir_col in dir_cols:
        i = dir_col.split("_")[-1]
        dif_col = f"ASWDIFD_cluster_{i}"
        if dif_col in df_s.columns:
            solar_data[f"sw_dir_cluster_{i}"] = df_s[dir_col].to_numpy()
            solar_data[f"sw_dif_cluster_{i}"] = df_s[dif_col].to_numpy()

    if not solar_data:
        raise ValueError("Function build_dwd_features: no matching ASWDIR/ASWDIFD cluster pairs found.")

    df_s_base = df_s[["date", "mtu"]].copy()
    df_solar_vals = pd.DataFrame(solar_data, index=df_s.index)
    df_s_full = pd.concat([df_s_base, df_solar_vals], axis=1)

    solar_cols = list(df_solar_vals.columns)

    df_solar_pivot = df_s_full.pivot_table(
        index="date",
        columns="mtu",
        values=solar_cols,
        aggfunc="mean",
    )
    df_solar_pivot.columns = [
        f"{col[0]}_mtu{int(col[1]):02d}" for col in df_solar_pivot.columns
    ]
    df_solar_pivot = df_solar_pivot.interpolate(axis=1, limit=4, limit_area="inside")

    # ------------------------------------------------------------------
    # Combine and convert index to timezone-aware midnight timestamps
    # ------------------------------------------------------------------
    df_pivot = pd.concat([df_wind_pivot, df_solar_pivot], axis=1)

    # Ensure that every calendar day between min and max has a row.
    # Days without data (e.g. due to a missing DWD folder) receive NaN rows
    # and are later removed by dropna=True in merge_all_features.
    full_index = pd.date_range(
        start=df_pivot.index.min(),
        end=df_pivot.index.max(),
        freq="D",
    )
    df_pivot = df_pivot.reindex(full_index)

    df_pivot.index = pd.to_datetime(df_pivot.index).tz_localize(tz_local)
    df_pivot.index.name = "date"
    return df_pivot

## Price Features

In [ ]:
def build_price_features(
    df_prices: pd.DataFrame,
    df_prices_exaa_15: Optional[pd.DataFrame] = None,
    exaa_vector: bool = False,
    exaa_only: bool = False,
) -> pd.DataFrame:
    """
    Build price-based feature blocks for the LEAR model (one row per calendar day).

    Constructs lagged daily price vectors (d-1, d-2, d-7), each as a flat
    96-MTU daily profile. Optionally includes an EXAA d0 vector, or builds
    an EXAA-only feature matrix without lagged price vectors.

    DST handling
    ------------
    Lags are computed in UTC (shift(freq='1D')) to avoid DST ambiguity.
    Pivots use aggfunc='mean' for clock-back days (25h) and interpolate
    (limit=4, inside) to impute the 4 missing MTU slots on clock-forward
    days (23h). All matrices are reindexed to a fixed 96-column layout.

    Parameters
    ----------
    df_prices : pd.DataFrame
        15-minute SDAC price series with column 'price_da' [EUR/MWh].
    df_prices_exaa_15 : pd.DataFrame, optional
        15-minute EXAA price series. Required if exaa_vector=True or exaa_only=True.
    exaa_vector : bool
        Include the d0 EXAA daily price vector alongside lagged vectors.
        Defaults to False.
    exaa_only : bool
        If True, return only the EXAA d0 vector without any lagged price vectors.
        Implies exaa_vector=True. Defaults to False.

    Returns
    -------
    pd.DataFrame
        Feature matrix with one row per calendar day (Midnight Europe/Berlin).
    """
    if exaa_only:
        exaa_vector = True

    if exaa_vector and df_prices_exaa_15 is None:
        raise ValueError("df_prices_exaa_15 is required when exaa_vector=True or exaa_only=True.")

    df = df_prices.copy().sort_index()
    df_utc = df.tz_convert("UTC")

    # ------------------------------------------------------------------
    # 1) Time-based lag features (computed in UTC to avoid DST ambiguity)
    # ------------------------------------------------------------------
    df_utc["price_lag1d"] = df_utc["price_da"].shift(freq="1D")
    df_utc["price_lag2d"] = df_utc["price_da"].shift(freq="2D")
    df_utc["price_lag7d"] = df_utc["price_da"].shift(freq="7D")

    df["price_lag1d"] = df_utc["price_lag1d"].values
    df["price_lag2d"] = df_utc["price_lag2d"].values
    df["price_lag7d"] = df_utc["price_lag7d"].values

    # ------------------------------------------------------------------
    # 2) Helper: pivot price series to daily 96-MTU matrix.
    # ------------------------------------------------------------------
    def _build_daily_matrix(series: pd.Series, col_name: str) -> pd.DataFrame:
        df_tmp = series.to_frame(col_name).copy()
        df_tmp["date_local"] = df_tmp.index.floor("D")
        df_tmp["mtu"] = df_tmp.index.hour * 4 + df_tmp.index.minute // 15
        return (
            df_tmp
            .pivot_table(
                index="date_local",
                columns="mtu",
                values=col_name,
                aggfunc="mean",
            )
            .reindex(columns=range(96))
            .interpolate(axis=1, limit=4, limit_area="inside")
        )

    daily_vector_dfs = []

    # ------------------------------------------------------------------
    # 3) Daily price vectors (d-1, d-2, d-7) -- skipped if exaa_only
    # ------------------------------------------------------------------
    if not exaa_only:
        daily_matrix_da = _build_daily_matrix(df["price_da"], "price_da")

        for lag, prefix in [(1, "price_d1"), (2, "price_d2"), (7, "price_d7")]:
            dm = daily_matrix_da.shift(lag)
            dm.columns = [f"{prefix}_mtu_{int(c):02d}" for c in dm.columns]
            daily_vector_dfs.append(dm)

    # ------------------------------------------------------------------
    # 4) EXAA daily vector (target day d0, ex ante available)
    # ------------------------------------------------------------------
    if exaa_vector:
        daily_matrix_exaa = _build_daily_matrix(
            df_prices_exaa_15["price_exaa"], "price_exaa"
        )
        dm = daily_matrix_exaa.copy()
        dm.columns = [f"exaa_d0_mtu_{int(c):02d}" for c in dm.columns]
        daily_vector_dfs.append(dm)

    result = pd.concat(daily_vector_dfs, axis=1, sort=False)
    result.index.name = "date"
    return result

## Load Features

In [ ]:
def build_load_features(
    df_load_fc: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build load forecast feature block for the LEAR model (one row per calendar day).

    The full 96-MTU day-ahead load forecast profile of the target day (d0)
    is returned as a flat feature vector per day.

    DST handling
    ------------
    Pivot uses aggfunc='mean' for clock-back days (25h) and interpolate
    (limit=4, inside) to impute the 4 missing MTU slots on clock-forward
    days (23h). Output is reindexed to a fixed 96-column layout.

    Parameters
    ----------
    df_load_fc : pd.DataFrame
        15-minute load forecast series with column 'load_fc' [MW].

    Returns
    -------
    pd.DataFrame
        Feature matrix with one row per calendar day (Midnight Europe/Berlin),
        columns 'load_d0_mtu_{mm}'.
    """
    df = df_load_fc.copy().sort_index()
    df["date_local"] = df.index.floor("D")
    df["mtu"] = df.index.hour * 4 + df.index.minute // 15

    daily_matrix = (
        df
        .pivot_table(
            index="date_local",
            columns="mtu",
            values="load_fc",
            aggfunc="mean",
        )
        .reindex(columns=range(96))
        .interpolate(axis=1, limit=4, limit_area="inside")
    )
    daily_matrix.columns = [f"load_d0_mtu_{int(c):02d}" for c in daily_matrix.columns]
    daily_matrix.index.name = "date"
    return daily_matrix

## Calendar Features

In [ ]:
def build_temporal_features(
    daily_index: pd.DatetimeIndex,
) -> pd.DataFrame:
    """
    Build temporal features for the LEAR model (one row per calendar day).

    Includes a market regime indicator, weekday one-hot encoding, and
    a public holiday indicator for Germany.

    Parameters
    ----------
    daily_index : pd.DatetimeIndex
        Timezone-aware Midnight timestamps (Europe/Berlin), one per calendar day.

    Returns
    -------
    pd.DataFrame
        Feature matrix indexed by daily_index with columns:
        'is_15min_market', 'weekday_0' ... 'weekday_6', 'is_holiday'.
    """
    if not isinstance(daily_index, pd.DatetimeIndex):
        raise TypeError("daily_index must be a DatetimeIndex.")

    regime_ts = POST_REGIME_START.tz_convert(daily_index.tz)

    df_out = pd.DataFrame(index=daily_index)

    # Market regime indicator (0 = hourly, 1 = 15-min)
    df_out["is_15min_market"] = (daily_index >= regime_ts).astype(int)

    # Weekday one-hot encoding (0=Mon, ..., 6=Sun)
    weekday_oh = pd.get_dummies(daily_index.weekday, prefix="weekday", dtype=int)
    weekday_oh.index = daily_index
    df_out = pd.concat([df_out, weekday_oh], axis=1)

    # Public holiday indicator (Germany)
    de_holidays = holidays.Germany(years=daily_index.year.unique())
    df_out["is_holiday"] = pd.Series(daily_index.date, index=daily_index).isin(set(de_holidays.keys())).astype(int)

    df_out.index.name = "date"
    return df_out

## Feature Assembly

In [ ]:
def merge_all_features(
    df_weather_features: pd.DataFrame,
    df_price_features: pd.DataFrame,
    df_load_features: pd.DataFrame,
    df_time_features: pd.DataFrame,
    dropna: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Merge all engineered feature blocks into one daily dataset.

    All feature blocks must have a daily index (one row per calendar day,
    Midnight Europe/Berlin timezone-aware timestamps). Returns X_daily with
    N_days rows and n_features columns.

    Parameters
    ----------
    df_weather_features : pd.DataFrame
        Weather feature block as returned by build_era5_features() or
        build_dwd_features().
    df_price_features : pd.DataFrame
        Price feature block as returned by build_price_features().
    df_load_features : pd.DataFrame
        Load feature block as returned by build_load_features().
    df_time_features : pd.DataFrame
        Temporal feature block as returned by build_temporal_features().
    dropna : bool
        If True, rows containing any NaN are dropped. Defaults to True.

    Returns
    -------
    X_daily : pd.DataFrame
        Merged feature matrix at daily resolution (one row per calendar day).
    dropped_info : pd.DataFrame
        Table of dropped dates and their NaN columns, for debugging.
    """
    X = (
        df_weather_features
        .join(df_price_features, how="inner")
        .join(df_load_features, how="inner")
        .join(df_time_features, how="inner")
        .sort_index()
    )

    # Track and optionally drop NaN rows
    nan_mask = X.isna().any(axis=1)

    dropped_info = pd.DataFrame({
        "date": X.index[nan_mask],
        "nan_columns": X.loc[nan_mask].isna().apply(
            lambda row: row.index[row].tolist(), axis=1
        ).values,
    })

    if dropna:
        X = X.loc[~nan_mask]

    X.index.name = "date"
    return X, dropped_info

## Target Construction

In [ ]:
def build_y_matrix(
    df_prices_15: pd.DataFrame,
    daily_index: pd.DatetimeIndex,
) -> pd.DataFrame:
    """
    Build the target matrix Y with shape (N_days x 96).

    Index: timezone-aware Midnight-Timestamps (Europe/Berlin), identical to X_daily.index.
    Columns: Integer 0..95 (MTU index within the day).

    Pre-regime (hourly prices): MTUs 1-3, 5-7, ... have NaN after pivot and are
    filled via interpolate(axis=1, limit=4, inside), giving all 4 quarter-hour
    slots of each hour the same price. This replicates the old merge_all_features
    forward-fill behavior.

    Post-regime (native 15-min): directly pivoted.

    DST handling
    ------------
    aggfunc='mean' handles clock-back days (25h, duplicate MTU slots).
    interpolate(limit=4, inside) fills the 4 missing slots on clock-forward
    days (23h).

    Parameters
    ----------
    df_prices_15 : pd.DataFrame
        15-minute SDAC price series with column 'price_da' [EUR/MWh] and
        timezone-aware index (Europe/Berlin).
    daily_index : pd.DatetimeIndex
        Midnight timestamps that define which days to include. Typically
        X_daily.index.

    Returns
    -------
    pd.DataFrame
        Y matrix with shape (N_days, 96). NaN rows for days not in df_prices_15.
    """
    df = df_prices_15.copy().sort_index()
    df["date_local"] = df.index.floor("D")
    df["mtu"] = df.index.hour * 4 + df.index.minute // 15

    Y = (
        df.pivot_table(
            index="date_local",
            columns="mtu",
            values="price_da",
            aggfunc="mean",
        )
        .reindex(columns=range(96))
        .interpolate(axis=1, limit=4, limit_area="inside")
    )
    Y.columns = range(96)

    # Align to the provided daily index
    Y = Y.reindex(daily_index)
    return Y

# LEAR Operational Model
---

## Operational Model

### Scaling

In [ ]:
from scipy.stats import norm

# Precompute the 0.75 quantile of the standard normal distribution
Z_075 = norm.ppf(0.75)

In [ ]:
def robust_params(x, eps: float = 1e-12) -> tuple[float, float]:
    """
    Compute robust centering and scaling parameters.

    Instead of mean and standard deviation, we use the median and
    the Median Absolute Deviation (MAD), which are more robust to
    extreme electricity price spikes.

    Parameters
    ----------
    x : array-like
        Input data.
    eps : float
        Minimum scale to prevent division by zero. Defaults to 1e-12.

    Returns
    -------
    center : float
        Sample median.
    scale : float
        MAD-based scale estimate (MAD / z_0.75).
    """
    x = np.asarray(x, dtype=float)

    # Robust center: sample median
    center = np.nanmedian(x)

    # Median absolute deviation
    mad = np.nanmedian(np.abs(x - center))

    # Convert MAD to a standard-deviation-like scale
    scale = mad / Z_075

    # Prevent division by zero or numerical instability
    if not np.isfinite(scale) or scale <= eps:
        scale = 1.0

    return float(center), float(scale)


def inverse_vst_bias_corrected(
    y_hat_trans,
    residuals_trans,
    center: float,
    scale: float,
):
    """
    Map predictions from the transformed space back to the original scale.

    The inverse transformation applies the sinh function and includes
    an empirical bias correction using in-sample residuals. This follows
    the approach commonly used in electricity price forecasting literature.

    Parameters
    ----------
    y_hat_trans : array-like
        Predictions in the transformed space.
    residuals_trans : array-like
        In-sample residuals in the transformed space.
    center : float
        Robust center as returned by robust_params().
    scale : float
        Robust scale as returned by robust_params().

    Returns
    -------
    np.ndarray
        Bias-corrected predictions in the original price space.
    """
    y_hat_trans = np.asarray(y_hat_trans, dtype=float).reshape(-1)
    residuals_trans = np.asarray(residuals_trans, dtype=float).reshape(-1)

    # If no residuals are available, apply simple inverse transformation
    if residuals_trans.size == 0:
        return center + scale * np.sinh(y_hat_trans)

    # Empirical bias correction
    return center + scale * np.mean(
        np.sinh(y_hat_trans[:, None] + residuals_trans[None, :]),
        axis=1,
    )

In [ ]:
def scale_fold_point(
    X_tr: pd.DataFrame,
    X_va: pd.DataFrame,
    y_tr: pd.Series,
    y_va: Optional[pd.Series],
    use_vst: bool = True,
    ssrd_filter_min_range: float = 20.0,
    ssrd_filter_min_pos_share: float = 0.50,
    ssrd_filter_min_iqr: Optional[float] = None,
):
    """
    Apply fold-wise feature and target scaling for the LEAR point model.

    Scaling strategy per feature group:

    - price: robust scaling (median/MAD) + arcsinh
    - wind: robust scaling only (no arcsinh)
    - load: robust scaling only (no arcsinh)
    - SSRD, sw_dir, sw_dif, other continuous: MinMaxScaler
    - time / dummy: unchanged

    Solar columns (SSRD, sw_dir, sw_dif) are filtered for degeneracy on the
    training fold before scaling. Degenerate columns are zeroed out.

    Parameters
    ----------
    X_tr : pd.DataFrame
        Training feature matrix.
    X_va : pd.DataFrame
        Validation (or test) feature matrix.
    y_tr : pd.Series
        Training target values.
    y_va : pd.Series, optional
        Validation target values. If None, y_va_scaled is returned as None.
    use_vst : bool
        If True, apply arcsinh variance-stabilizing transformation to the target.
        Defaults to True.
    ssrd_filter_min_range : float
        Minimum value range for a solar column to be considered non-degenerate.
        Defaults to 20.0.
    ssrd_filter_min_pos_share : float
        Minimum share of positive values for a solar column to be considered
        non-degenerate. Defaults to 0.50.
    ssrd_filter_min_iqr : float, optional
        Minimum IQR threshold for solar degeneracy filtering. If None, IQR
        filtering is skipped.

    Returns
    -------
    X_tr_scaled : pd.DataFrame
        Scaled training feature matrix.
    X_va_scaled : pd.DataFrame
        Scaled validation feature matrix.
    y_tr_scaled : np.ndarray
        Scaled (and optionally VST-transformed) training target.
    y_va_scaled : np.ndarray or None
        Scaled validation target, or None if y_va was None.
    y_params : dict
        Target scaling parameters: y_center, y_scale, use_vst,
        and degenerate_ssrd_cols.
    """
    cols = X_tr.columns.tolist()

    price_cols   = [c for c in cols if c.startswith(("price_d", "exaa_d"))]
    time_cols    = [c for c in cols if c.startswith("weekday_")
                    or c in ["is_15min_market", "is_holiday"]]
    ssrd_cols    = [c for c in cols if "ssrd" in c.lower()]
    sw_dir_cols  = [c for c in cols if "sw_dir" in c.lower()]
    sw_dif_cols  = [c for c in cols if "sw_dif" in c.lower()]
    wind_cols    = [c for c in cols if "wind_speed" in c.lower()]
    load_cols    = [c for c in cols if "load" in c.lower()]
    other_cols   = [c for c in cols if c not in
                    price_cols + time_cols + ssrd_cols + sw_dir_cols + sw_dif_cols + wind_cols + load_cols]

    # ------------------------------------------------------------
    # 0) Training-fold SSRD filtering
    # ------------------------------------------------------------
    X_tr_work = X_tr.copy()
    X_va_work = X_va.copy()

    degenerate_ssrd_cols = []

    for col in ssrd_cols + sw_dir_cols + sw_dif_cols:
        x = X_tr_work[col].astype(float)
        col_range = x.max() - x.min()
        pos_share = (x > 0).mean()
        is_degenerate = (col_range < ssrd_filter_min_range) or (
            pos_share < ssrd_filter_min_pos_share
        )
        if ssrd_filter_min_iqr is not None:
            col_iqr = x.quantile(0.75) - x.quantile(0.25)
            is_degenerate = is_degenerate or (col_iqr < ssrd_filter_min_iqr)
        if is_degenerate:
            degenerate_ssrd_cols.append(col)

    if degenerate_ssrd_cols:
        X_tr_work.loc[:, degenerate_ssrd_cols] = 0.0
        X_va_work.loc[:, degenerate_ssrd_cols] = 0.0

    # ------------------------------------------------------------
    # 1) Target scaling
    # ------------------------------------------------------------
    y_center, y_scale = robust_params(y_tr.values)
    y_tr_scaled = (y_tr.values - y_center) / y_scale
    y_va_scaled = (y_va.values - y_center) / y_scale if y_va is not None else None

    if use_vst:
        y_tr_scaled = np.arcsinh(y_tr_scaled)
        if y_va_scaled is not None:
            y_va_scaled = np.arcsinh(y_va_scaled)

    y_params = {
        "y_center": float(y_center),
        "y_scale": float(y_scale),
        "use_vst": bool(use_vst),
        "degenerate_ssrd_cols": degenerate_ssrd_cols,
    }

    # ------------------------------------------------------------
    # 2) Feature scaling
    # ------------------------------------------------------------
    X_tr_scaled = X_tr_work.copy()
    X_va_scaled = X_va_work.copy()

    # --- price: robust scaling + arcsinh
    for col in price_cols:
        center, scale = robust_params(X_tr_work[col].values)
        X_tr_scaled[col] = np.arcsinh((X_tr_work[col].values - center) / scale)
        X_va_scaled[col] = np.arcsinh((X_va_work[col].values - center) / scale)

    # --- wind: robust scaling only (no arcsinh)
    for col in wind_cols:
        center, scale = robust_params(X_tr_work[col].values)
        X_tr_scaled[col] = (X_tr_work[col].values - center) / scale
        X_va_scaled[col] = (X_va_work[col].values - center) / scale

    # --- load: robust scaling only (no arcsinh)
    for col in load_cols:
        center, scale = robust_params(X_tr_work[col].values)
        X_tr_scaled[col] = (X_tr_work[col].values - center) / scale
        X_va_scaled[col] = (X_va_work[col].values - center) / scale

    # --- SSRD + sw_dir + sw_dif + other continuous: MinMaxScaler
    cont_cols = ssrd_cols + sw_dir_cols + sw_dif_cols + other_cols
    if cont_cols:
        scaler = MinMaxScaler()
        X_tr_scaled[cont_cols] = scaler.fit_transform(X_tr_work[cont_cols])
        X_va_scaled[cont_cols] = scaler.transform(X_va_work[cont_cols])

    # --- time / dummy: unchanged

    return X_tr_scaled, X_va_scaled, y_tr_scaled, y_va_scaled, y_params

### Rolling Forecast Loop

In [ ]:
def rolling_point_forecast(
    X: pd.DataFrame,
    Y: pd.DataFrame,
    forecast_days: list[pd.Timestamp],
    train_days: int,
    use_vst: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Run rolling day-ahead LEAR point forecasts with MTU-specific models.

    X is the daily feature matrix (N_days x n_features).
    Y is the target matrix (N_days x 96), where columns 0..95 are MTUs.

    LassoLarsCV is used for forecast days >= LARS_START_DATE_OPERATIONAL
    (global constant), LassoCV otherwise.

    Parameters
    ----------
    X : pd.DataFrame
        Daily feature matrix with timezone-aware index (Europe/Berlin).
    Y : pd.DataFrame
        Daily target matrix with columns 0..95 (MTU index).
    forecast_days : list[pd.Timestamp]
        Ordered list of days to forecast.
    train_days : int
        Number of calendar days in the rolling training window.
    use_vst : bool
        If True, apply VST transformation to the target. Defaults to True.

    Returns
    -------
    forecast_df : pd.DataFrame
        y_pred and y_true per MTU-level timestamp.
    runtime_df : pd.DataFrame
        Per-day computation times.
    coef_df : pd.DataFrame
        Non-zero coefficients per day and MTU.
    intercept_df : pd.DataFrame
        Intercepts per day and MTU.
    degenerate_ssrd_df : pd.DataFrame
        Degenerate SSRD columns per day and MTU.
    """
    all_preds               = []
    all_trues               = []
    runtime_records         = []
    coef_records            = []
    intercept_records       = []
    degenerate_ssrd_records = []

    for day in forecast_days:
        day_start = time.perf_counter()

        use_lars = day >= LARS_START_DATE_OPERATIONAL

        train_start = day - pd.Timedelta(days=train_days)
        train_end   = day - pd.Timedelta(days=1)
        train_mask  = (X.index >= train_start) & (X.index <= train_end)
        test_mask   = X.index == day

        if train_mask.sum() == 0 or test_mask.sum() == 0:
            continue

        y_hat_day  = {}
        y_true_day = {}

        for mtu in range(96):
            X_tr = X.loc[train_mask]
            y_tr = Y.loc[train_mask, mtu]
            X_te = X.loc[test_mask]

            X_tr_s, X_te_s, y_tr_s, _, y_params = scale_fold_point(
                X_tr=X_tr, X_va=X_te,
                y_tr=y_tr, y_va=None,
                use_vst=use_vst,
            )

            if use_lars:
                model = LassoLarsCV(cv=5, max_iter=1000, n_jobs=1)
            else:
                model = LassoCV(cv=5, tol=1e-3, max_iter=10_000, n_jobs=1)

            model.fit(X_tr_s.values, y_tr_s)
            y_pred_s = model.predict(X_te_s.values)

            if use_vst:
                y_fit_s     = model.predict(X_tr_s.values)
                residuals_s = y_tr_s - y_fit_s
                y_pred = inverse_vst_bias_corrected(
                    y_hat_trans=y_pred_s,
                    residuals_trans=residuals_s,
                    center=y_params["y_center"],
                    scale=y_params["y_scale"],
                )
            else:
                y_pred = y_params["y_center"] + y_params["y_scale"] * y_pred_s

            mtu_timestamp = day + pd.Timedelta(minutes=15 * mtu)
            y_hat_day[mtu_timestamp]  = float(np.asarray(y_pred).ravel()[0])
            y_true_day[mtu_timestamp] = float(Y.loc[day, mtu])

            nonzero_mask = model.coef_ != 0
            coef_records.append({
                "forecast_day": day.date(),
                "mtu":          mtu,
                "alpha":        model.alpha_,
                "n_nonzero":    int(nonzero_mask.sum()),
                "nonzero_cols": X_tr.columns[nonzero_mask].tolist(),
                "nonzero_vals": model.coef_[nonzero_mask].tolist(),
                "use_lars":     use_lars,
            })

            intercept_records.append({
                "forecast_day": day.date(),
                "mtu":          mtu,
                "intercept":    float(model.intercept_),
            })

            degenerate_ssrd_records.append({
                "forecast_day":    day.date(),
                "mtu":             mtu,
                "n_degenerate":    len(y_params["degenerate_ssrd_cols"]),
                "degenerate_cols": y_params["degenerate_ssrd_cols"],
            })

        all_preds.append(pd.Series(y_hat_day, dtype=float))
        all_trues.append(pd.Series(y_true_day, dtype=float))

        day_runtime = time.perf_counter() - day_start
        runtime_records.append({
            "forecast_day":    day,
            "train_days":      train_days,
            "use_vst":         use_vst,
            "use_lars":        use_lars,
            "runtime_seconds": day_runtime,
        })
        print(f"  {day.date()}  {'LARS' if use_lars else 'LassoCV'}  {day_runtime:.1f}s")

    y_pred_all = pd.concat(all_preds).sort_index()
    y_true_all = pd.concat(all_trues).sort_index()

    forecast_df = pd.DataFrame({
        "y_pred": y_pred_all,
        "y_true": y_true_all,
    })
    runtime_df         = pd.DataFrame(runtime_records)
    coef_df            = pd.DataFrame(coef_records)
    intercept_df       = pd.DataFrame(intercept_records)
    degenerate_ssrd_df = pd.DataFrame(degenerate_ssrd_records)

    return forecast_df, runtime_df, coef_df, intercept_df, degenerate_ssrd_df

### Save Experiment Outputs

In [ ]:
def compute_metrics(fc: pd.DataFrame, label: str) -> dict:
    """
    Compute point forecast error metrics for a given forecast slice.

    Parameters
    ----------
    fc : pd.DataFrame
        Forecast DataFrame with columns 'y_pred' and 'y_true'.
    label : str
        Period label used as identifier in the output dict.

    Returns
    -------
    dict
        Dictionary with keys: period, mae, rmse, bias, n_obs, n_inf_nan.
    """
    n_inf_nan = int((~np.isfinite(fc["y_pred"])).sum())
    valid = fc[np.isfinite(fc["y_pred"])]

    mae  = mean_absolute_error(valid["y_true"], valid["y_pred"]) if len(valid) > 0 else np.nan
    rmse = np.sqrt(((valid["y_true"] - valid["y_pred"]) ** 2).mean()) if len(valid) > 0 else np.nan
    bias = (valid["y_pred"] - valid["y_true"]).mean() if len(valid) > 0 else np.nan

    return {
        "period":    label,
        "mae":       mae,
        "rmse":      rmse,
        "bias":      bias,
        "n_obs":     len(fc),
        "n_inf_nan": n_inf_nan,
    }

In [ ]:
def save_experiment_outputs(
    name: str,
    forecast_df: pd.DataFrame,
    runtime_df: pd.DataFrame,
    config: dict,
    export_dir: Path,
):
    """
    Save all experiment outputs to disk, including monthly metrics.
    Parameters
    ----------
    name : str
        Experiment name stored in config.json.
    forecast_df : pd.DataFrame
        Forecast DataFrame with columns 'y_pred' and 'y_true'.
    runtime_df : pd.DataFrame
        Per-day runtime records.
    config : dict
        Experiment configuration. Extended in-place with summary statistics.
    export_dir : Path
        Output directory. Created automatically if it does not exist.
    Returns
    -------
    None
    """
    export_dir.mkdir(parents=True, exist_ok=True)
    forecast_df.to_csv(export_dir / "forecast.csv", index=True)
    runtime_df.to_csv(export_dir / "runtime.csv", index=False)
    # Compute metrics for full period and each calendar month
    rows = [compute_metrics(forecast_df, "full")]
    months = forecast_df.index.tz_localize(None).to_period("M").unique()
    for period in months:
        mask = forecast_df.index.tz_localize(None).to_period("M") == period
        rows.append(compute_metrics(forecast_df[mask], str(period)))
    pd.DataFrame(rows).to_csv(export_dir / "metrics.csv", index=False)
    with open(export_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"✓ Saved: forecast | runtime | metrics | config  →  {export_dir}")

### Evaluation: Plot & Metrics

In [ ]:
def evaluate_and_plot_forecast_from_df(
    forecasts: pd.DataFrame,
    start: str,
    end: str,
    model_label: str = "Forecast",
    title: str = "Rolling Point Forecast vs Actual",
    figsize: tuple = (14, 5),
):
    """
    Evaluate and plot a rolling point forecast against actual values.

    Prints MAE, RMSE, and Bias for the specified evaluation window
    and displays a time series plot.

    Parameters
    ----------
    forecasts : pd.DataFrame
        Forecast DataFrame with columns 'y_pred' and 'y_true' and a
        timezone-aware DatetimeIndex.
    start : str
        Start of the evaluation window (date string, e.g. '2025-10-01').
    end : str
        End of the evaluation window (date string, e.g. '2025-11-30').
    model_label : str
        Legend label for the forecast line. Defaults to 'Forecast'.
    title : str
        Plot title. Defaults to 'Rolling Point Forecast vs Actual'.
    figsize : tuple
        Figure size. Defaults to (14, 5).

    Returns
    -------
    dict
        Dictionary with keys 'MAE', 'RMSE', 'Bias'.
    """
    tz = forecasts.index.tz
    start = pd.Timestamp(start, tz=tz)
    end = pd.Timestamp(end, tz=tz) + pd.Timedelta(days=1) - pd.Timedelta(minutes=15)

    df = forecasts.loc[start:end]

    if df.empty:
        raise ValueError(f"No data found in the evaluation window {start} to {end}.")

    missing_cols = {"y_true", "y_pred"} - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    metrics = compute_metrics(df, label="eval")
    mae  = metrics["mae"]
    rmse = metrics["rmse"]
    bias = metrics["bias"]

    print(f"Evaluation window: {start} → {end}")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"Bias : {bias:.3f}")

    plt.figure(figsize=figsize)
    plt.plot(df["y_true"].index, df["y_true"].values, label="Actual", alpha=0.8)
    plt.plot(df["y_pred"].index, df["y_pred"].values, label=model_label, alpha=0.8)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

    return {"MAE": mae, "RMSE": rmse, "Bias": bias}

## **Execution**

#### Load Input Data

In [ ]:
# -- Data --

df_prices_15 = fetch_prices(ENTSOE_START_DATE, ENTSOE_END_DATE)
df_prices_exaa_15 = fetch_prices_exaa(ENTSOE_START_DATE, ENTSOE_END_DATE)
df_load_fc_entso_15 = fetch_load_forecast(ENTSOE_START_DATE, ENTSOE_END_DATE)
df_era5 = load_era5(
    dirs=[ERA5_2024_DIR, ERA5_2025_DIR, ERA5_2026_DIR],
)
df_dwd_hourly, df_dwd_qh = load_dwd(
    icon_dir=ICON_DIR,
    start_folder_date=START_FOLDER_DATE,
    required_run=REQUIRED_RUN,
    skip_dates=SKIP_DATES,
)

#### Build Feature Matrix and Target

In [ ]:
# -- Feature Engineering --

if WEATHER_SOURCE_OP == "ERA5":
    df_weather_features = build_era5_features(df_era5=df_era5)
elif WEATHER_SOURCE_OP == "DWD":
    df_weather_features = build_dwd_features(
        df_hourly=df_dwd_hourly,
        df_qh=df_dwd_qh,
    )
else:
    raise ValueError(f"Unknown weather source: {WEATHER_SOURCE_OP}. Choose 'ERA5' or 'DWD'.")

In [ ]:
# -- Feature and Target Assembly --

## Build daily_index for temporal features (all unique midnight timestamps from price data)
daily_index = df_prices_15.index.normalize().unique().sort_values()

## Build daily feature matrix
if USE_EXAA_ONLY_OP:
    X_lear_op = build_price_features(
        df_prices=df_prices_15,
        exaa_only=True,
        df_prices_exaa_15=df_prices_exaa_15,
    )
    dropped_X_lear_op = pd.DataFrame()
else:
    X_lear_op, dropped_X_lear_op = merge_all_features(
        df_weather_features=df_weather_features,
        df_price_features=build_price_features(
            df_prices=df_prices_15,
            exaa_vector=USE_EXAA_OP,
            df_prices_exaa_15=df_prices_exaa_15 if USE_EXAA_OP else None,
        ),
        df_load_features=build_load_features(df_load_fc_entso_15),
        df_time_features=build_temporal_features(daily_index),
    )

Y_lear_op = build_y_matrix(df_prices_15, X_lear_op.index)

In [ ]:
# Sanity Check
print(f"Number of dropped rows (days): {len(dropped_X_lear_op)}")
X_lear_op.shape
display(dropped_X_lear_op)

In [ ]:
# Align feature matrix and target (drop rows where X has NaN; Y is built independently)
valid_mask = X_lear_op.notna().all(axis=1)
X_lear_op = X_lear_op.loc[valid_mask]
Y_lear_op = Y_lear_op.loc[valid_mask]

print(f"X shape    : {X_lear_op.shape}")
print(f"Y shape    : {Y_lear_op.shape}")
print(f"Date range : {X_lear_op.index.min().date()} → {X_lear_op.index.max().date()}")

#### Define Forecast Window

In [ ]:
# Define forecast days for the rolling evaluation period

forecast_days = pd.date_range(
    start=TEST_START_P.normalize(),
    end=TEST_END_P.normalize(),
    freq="D",
    tz="Europe/Berlin",
)

# Quick sanity check
print(f"Number of forecast days: {len(forecast_days)}")
print(f"First forecast day: {forecast_days[0]}")
print(f"Last forecast day:  {forecast_days[-1]}")

#### Run Rolling Forecast Loop

In [ ]:
fc, rt, cf, ic, deg = rolling_point_forecast(
    X=X_lear_op,
    Y=Y_lear_op,
    forecast_days=forecast_days,
    train_days=TRAIN_DAYS_ROLLING_P,
    use_vst=VST_BOOLEAN,
)

In [ ]:
# Sanity Check
metrics = compute_metrics(fc, label="full")
n_invalid = int((~np.isfinite(fc["y_pred"])).sum())

print(f"{'Observations':<14}: {len(fc)}")
print(f"{'Invalid preds':<14}: {n_invalid}")
print(f"{'MAE':<14}: {metrics['mae']:.3f}")
print(f"{'RMSE':<14}: {metrics['rmse']:.3f}")
print(f"{'Bias':<14}: {metrics['bias']:.3f}")

#### Save Experiment Outputs

In [ ]:
save_experiment_outputs(
    name=EXPERIMENT_NAME_OP,
    forecast_df=fc,
    runtime_df=rt,
    config={
        "experiment_name": EXPERIMENT_NAME_OP,
        "use_exaa":        USE_EXAA_OP,
        "use_exaa_only":   USE_EXAA_ONLY_OP,
        "n_clusters":      None if USE_EXAA_ONLY_OP else N_CLUSTERS_OP,
        "weather_source":  None if USE_EXAA_ONLY_OP else WEATHER_SOURCE_OP,
        "train_days":      TRAIN_DAYS_ROLLING_P,
        "use_vst":         VST_BOOLEAN,
        "lars_start_date": str(LARS_START_DATE_OPERATIONAL.date()),
        "test_start":      str(TEST_START_P.date()),
        "test_end":        str(TEST_END_P.date()),
    },
    export_dir=EXPORT_BASE_P,
)

#### Quick Evaluation: Plot and Metrics

In [ ]:
evaluate_and_plot_forecast_from_df(
    forecasts=fc,
    start=str(TEST_START_P.date()),
    end=str(TEST_END_P.date()),
    model_label=EXPERIMENT_NAME_OP,
    title=f"Rolling Point Forecast vs Actual Prices - {EXPERIMENT_NAME_OP}",
)

# LEAR ANC Model 
---

## ANC Model

### Scaling (min/max)

In [ ]:
def scale_fold_anc(
    X_tr: pd.DataFrame,
    X_te: pd.DataFrame,
    ssrd_filter_min_range: float = 20.0,
    ssrd_filter_min_pos_share: float = 0.50,
    ssrd_filter_min_iqr: Optional[float] = None,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Apply fold-wise Min-Max scaling for ANC re-estimation.

    Scaling strategy per feature group:

    - Calendar / dummy features: kept unchanged
    - SSRD, sw_dir, sw_dif features: training-fold-based degeneracy filter
      applied, degenerate columns are zeroed out before scaling
    - All other features: scaled to [0, 1] using training-fold statistics
    - Scaling is fit on the training fold only (no leakage)
    - Constant features are safely handled

    Parameters
    ----------
    X_tr : pd.DataFrame
        Training feature matrix.
    X_te : pd.DataFrame
        Test feature matrix.
    ssrd_filter_min_range : float
        Minimum value range for a solar column to be considered non-degenerate.
        Defaults to 20.0.
    ssrd_filter_min_pos_share : float
        Minimum share of positive values for a solar column to be considered
        non-degenerate. Defaults to 0.50.
    ssrd_filter_min_iqr : float, optional
        Minimum IQR threshold for solar degeneracy filtering. If None, IQR
        filtering is skipped.

    Returns
    -------
    X_tr_scaled : pd.DataFrame
        Scaled training feature matrix.
    X_te_scaled : pd.DataFrame
        Scaled test feature matrix.
    scaling_params : dict
        Per-column min, max, and range used for scaling, plus
        degenerate_ssrd_cols.
    """
    cols = X_tr.columns.tolist()

    calendar_cols = [
        c for c in cols
        if c.startswith("weekday_")
        or c in ["is_15min_market", "is_holiday"]
    ]
    ssrd_cols   = [c for c in cols if "ssrd" in c.lower()]
    sw_dir_cols = [c for c in cols if "sw_dir" in c.lower()]
    sw_dif_cols = [c for c in cols if "sw_dif" in c.lower()]
    scale_cols  = [c for c in cols if c not in calendar_cols]

    X_tr_work = X_tr.copy()
    X_te_work = X_te.copy()

    # --------------------------------------------------
    # Training-fold SSRD filtering
    # --------------------------------------------------
    degenerate_ssrd_cols = []

    for col in ssrd_cols + sw_dir_cols + sw_dif_cols:
        x = X_tr_work[col].astype(float)
        col_range = x.max() - x.min()
        pos_share = (x > 0).mean()
        is_degenerate = (col_range < ssrd_filter_min_range) or (
            pos_share < ssrd_filter_min_pos_share
        )
        if ssrd_filter_min_iqr is not None:
            col_iqr = x.quantile(0.75) - x.quantile(0.25)
            is_degenerate = is_degenerate or (col_iqr < ssrd_filter_min_iqr)
        if is_degenerate:
            degenerate_ssrd_cols.append(col)

    if degenerate_ssrd_cols:
        X_tr_work.loc[:, degenerate_ssrd_cols] = 0.0
        X_te_work.loc[:, degenerate_ssrd_cols] = 0.0

    # --------------------------------------------------
    # Min-Max scaling
    # --------------------------------------------------
    X_tr_scaled = X_tr_work.copy()
    X_te_scaled = X_te_work.copy()

    scaling_params = {"degenerate_ssrd_cols": degenerate_ssrd_cols}

    for col in scale_cols:
        col_min   = float(X_tr_work[col].min())
        col_max   = float(X_tr_work[col].max())
        col_range = col_max - col_min

        if not np.isfinite(col_range) or col_range <= 1e-12:
            X_tr_scaled[col] = 0.0
            X_te_scaled[col] = 0.0
            col_range = 1.0
        else:
            X_tr_scaled[col] = (X_tr_work[col] - col_min) / col_range
            X_te_scaled[col] = (X_te_work[col] - col_min) / col_range

        scaling_params[col] = {
            "min":   col_min,
            "max":   col_max,
            "range": float(col_range),
        }

    return X_tr_scaled, X_te_scaled, scaling_params

### Rolling Forecast Loop

In [ ]:
def rolling_anc_feature_importance(
    X: pd.DataFrame,
    Y: pd.DataFrame,
    forecast_days: list[pd.Timestamp],
    train_days: int,
) -> pd.DataFrame:
    """
    Run a rolling ANC estimation with MTU-specific LEAR models.

    X is the daily feature matrix (N_days x n_features).
    Y is the target matrix (N_days x 96), where columns 0..95 are MTUs.

    For each forecast day and each MTU, one separate LASSO model is re-estimated
    on a rolling calibration window using fold-wise Min-Max scaled features
    with training-fold-based SSRD degeneracy filtering.

    LassoLarsCV is used for forecast days >= LARS_START_DATE_ANC (global constant),
    LassoCV otherwise.

    Parameters
    ----------
    X : pd.DataFrame
        Daily feature matrix with timezone-aware index (Europe/Berlin).
    Y : pd.DataFrame
        Daily target matrix with columns 0..95 (MTU index).
    forecast_days : list[pd.Timestamp]
        Ordered list of days to forecast.
    train_days : int
        Number of calendar days in the rolling training window.

    Returns
    -------
    pd.DataFrame
        Long-format DataFrame with one row per forecast day, MTU, and feature.
        Columns: forecast_day, timestamp, mtu, train_days, use_lars, feature,
        feature_value, beta, contribution.
    """
    anc_records = []

    for day in forecast_days:
        use_lars = day >= LARS_START_DATE_ANC

        train_start = day - pd.Timedelta(days=train_days)
        train_end   = day - pd.Timedelta(days=1)
        train_mask  = (X.index >= train_start) & (X.index <= train_end)
        test_mask   = X.index == day

        if train_mask.sum() == 0 or test_mask.sum() == 0:
            continue

        for mtu in range(96):
            X_tr = X.loc[train_mask]
            y_tr = Y.loc[train_mask, mtu]
            X_te = X.loc[test_mask]

            X_tr_s, X_te_s, _ = scale_fold_anc(
                X_tr=X_tr,
                X_te=X_te,
            )

            if use_lars:
                model = LassoLarsCV(cv=5, max_iter=1000, n_jobs=1)
            else:
                model = LassoCV(cv=5, tol=1e-3, max_iter=10_000, n_jobs=1)

            model.fit(X_tr_s.values, y_tr.values)

            beta_series = pd.Series(model.coef_, index=X_tr_s.columns, dtype=float)

            mtu_timestamp = day + pd.Timedelta(minutes=15 * mtu)
            x_row = X_te_s.iloc[0]

            for feature in X_te_s.columns:
                feature_value = float(x_row[feature])
                beta          = float(beta_series[feature])
                contribution  = feature_value * beta

                anc_records.append({
                    "forecast_day":  day,
                    "timestamp":     mtu_timestamp,
                    "mtu":           mtu,
                    "train_days":    train_days,
                    "use_lars":      use_lars,
                    "feature":       feature,
                    "feature_value": feature_value,
                    "beta":          beta,
                    "contribution":  contribution,
                })

        print(f"  {day.date()}  {'LARS' if use_lars else 'LassoCV'}")

    return pd.DataFrame(anc_records)

## **Execution**

#### Load Input Data

In [ ]:
# -- Data --

df_prices_15 = fetch_prices(ENTSOE_START_DATE, ENTSOE_END_DATE)
df_prices_exaa_15 = fetch_prices_exaa(ENTSOE_START_DATE, ENTSOE_END_DATE)
df_load_fc_entso_15 = fetch_load_forecast(ENTSOE_START_DATE, ENTSOE_END_DATE)
df_era5 = load_era5(
    dirs=[ERA5_2024_DIR, ERA5_2025_DIR, ERA5_2026_DIR],
)
df_dwd_hourly, df_dwd_qh = load_dwd(
    icon_dir=ICON_DIR,
    start_folder_date=START_FOLDER_DATE,
    required_run=REQUIRED_RUN,
    skip_dates=SKIP_DATES,
)

#### Build Feature Matrix and Target

In [ ]:
# -- Feature Engineering --

if WEATHER_SOURCE_ANC == "ERA5":
    df_weather_features_anc = build_era5_features(df_era5=df_era5)
elif WEATHER_SOURCE_ANC == "DWD":
    df_weather_features_anc = build_dwd_features(
        df_hourly=df_dwd_hourly,
        df_qh=df_dwd_qh,
    )
else:
    raise ValueError(f"Unknown weather source: {WEATHER_SOURCE_ANC}. Choose 'ERA5' or 'DWD'.")

In [ ]:
# -- Feature Assembly --

## Build daily_index for temporal features (all unique midnight timestamps from price data)
daily_index = df_prices_15.index.normalize().unique().sort_values()

## Build daily feature matrix
if USE_EXAA_ONLY_ANC:
    X_lear_anc = build_price_features(
        df_prices=df_prices_15,
        exaa_only=True,
        df_prices_exaa_15=df_prices_exaa_15,
    )
    dropped_X_lear_anc = pd.DataFrame()
else:
    X_lear_anc, dropped_X_lear_anc = merge_all_features(
        df_weather_features=df_weather_features_anc,
        df_price_features=build_price_features(
            df_prices=df_prices_15,
            exaa_vector=USE_EXAA_ANC,
            df_prices_exaa_15=df_prices_exaa_15 if USE_EXAA_ANC else None,
        ),
        df_load_features=build_load_features(df_load_fc_entso_15),
        df_time_features=build_temporal_features(daily_index),
    )

Y_lear_anc = build_y_matrix(df_prices_15, X_lear_anc.index)

In [ ]:
# Sanity Check
print(f"Number of dropped rows (days): {len(dropped_X_lear_anc)}")
X_lear_anc.shape
display(dropped_X_lear_anc)

In [ ]:
# Build joint mask for complete cases (X only – Y is independently built)
valid_mask = X_lear_anc.notna().all(axis=1)

# Filter features and target
X_lear_anc = X_lear_anc.loc[valid_mask]
Y_lear_anc = Y_lear_anc.loc[valid_mask]

# Quick shape check
print(f"X shape: {X_lear_anc.shape}")
print(f"Y shape: {Y_lear_anc.shape}")

#### Define Forecast Window

In [ ]:
forecast_days_anc = pd.date_range(
    start=TEST_START_ANC.normalize(),
    end=TEST_END_ANC.normalize(),
    freq="D",
    tz=TARGET_TZ,
)

print(f"Number of forecast days : {len(forecast_days_anc)}")
print(f"First day               : {forecast_days_anc[0].date()}")
print(f"Last day                : {forecast_days_anc[-1].date()}")

#### Run Rolling Forecast Loop

In [ ]:
anc_df = rolling_anc_feature_importance(
    X=X_lear_anc,
    Y=Y_lear_anc,
    forecast_days=forecast_days_anc,
    train_days=TRAIN_DAYS_ROLLING_ANC,
)

print(f"ANC records : {len(anc_df)}")
print(f"Columns     : {anc_df.columns.tolist()}")

#### Overall Feature Importance:

In [ ]:
def map_feature_to_group(feature: str) -> str:
    """
    Map a raw regressor name to a high-level feature group
    used in the overall ANC analysis.

    Parameters
    ----------
    feature : str
        Raw feature column name from the LEAR model.

    Returns
    -------
    str
        High-level feature group label.
    """
    f = feature.lower()

    # -----------------------------
    # External market prices
    # -----------------------------
    if "exaa" in f:
        return "EXAA d"

    # -----------------------------
    # Lagged prices
    # -----------------------------
    if "price_d1" in f:
        return "Price d-1"

    if "price_d2" in f:
        return "Price d-2"

    if "price_d7" in f:
        return "Price d-7"

    # -----------------------------
    # Load
    # -----------------------------
    if "load_d0" in f:
        return "Load d"

    # -----------------------------
    # Weather
    # -----------------------------
    if "wind" in f:
        return "Wind d"

    if "ssrd" in f or "sw_dir" in f or "sw_dif" in f:
        return "Solar d"

    # -----------------------------
    # Calendar
    # -----------------------------
    if f.startswith("weekday_"):
        return "Weekday"

    if "is_holiday" in f:
        return "Holiday"

    if "is_15min_market" in f:
        return "15-min market dummy"

    # -----------------------------
    # Fallback
    # -----------------------------
    return "Other"

In [ ]:
# Apply mapping to ANC raw output
anc_df_analysis = anc_df.copy()
anc_df_analysis["feature_group"] = anc_df_analysis["feature"].apply(map_feature_to_group)

# Quick check
display(
    anc_df_analysis[["feature", "feature_group"]]
    .drop_duplicates()
    .sort_values(["feature_group", "feature"])
    .reset_index(drop=True)
)

In [ ]:
# Calculate ANC

# Step 1: grouped contribution per observation i and feature group j
grouped_contrib_df = (
    anc_df_analysis
    .groupby(
        ["forecast_day", "timestamp", "mtu", "train_days", "feature_group"],
        as_index=False
    )["contribution"]
    .sum()
    .rename(columns={"contribution": "group_contribution"})
)

# Step 2: absolute grouped contribution
grouped_contrib_df["abs_group_contribution"] = grouped_contrib_df["group_contribution"].abs()

# Step 3: ANC_j = average absolute grouped contribution over all observations
anc_summary_df = (
    grouped_contrib_df
    .groupby(["train_days", "feature_group"], as_index=False)["abs_group_contribution"]
    .mean()
    .rename(columns={"abs_group_contribution": "ANC"})
    .sort_values(["train_days", "ANC"], ascending=[True, False])
    .reset_index(drop=True)
)

display(anc_summary_df)

In [ ]:
# Plotting

anc_plot_df = (
    anc_summary_df.loc[anc_summary_df["train_days"] == TRAIN_DAYS_ROLLING_ANC]
    .sort_values("ANC", ascending=False)
    .sort_values("ANC", ascending=True)   # for horizontal bar plot
)

plt.figure(figsize=(7, 4))
plt.barh(anc_plot_df["feature_group"], anc_plot_df["ANC"])
plt.xlabel("ANC")
plt.ylabel("")
plt.title(f"Feature Groups by ANC (train_days={TRAIN_DAYS_ROLLING_ANC})")
plt.tight_layout()
plt.show()

#### Spatial/ Cluster Importance:

##### Cluster-Analysis for Wind:

In [ ]:
# Define Feature Groups

def map_wind_feature_to_cluster(feature: str) -> str:
    """
    Map a raw feature name to its wind cluster group.

    Parameters
    ----------
    feature : str
        Raw feature column name from the LEAR model.

    Returns
    -------
    str
        Wind cluster label (e.g. 'Wind Cluster 0'), or 'Other' if the
        feature does not belong to a wind cluster.
    """
    f = feature.lower()
    match = re.search(r'wind_speed_cluster_(\d+)_h\d+', f)
    if match:
        return f"Wind Cluster {match.group(1)}"
    return "Other"

# Apply mapping — keep only wind features
anc_df_wind = anc_df.copy()
anc_df_wind["cluster_group"] = anc_df_wind["feature"].apply(map_wind_feature_to_cluster)
anc_df_wind = anc_df_wind[anc_df_wind["cluster_group"] != "Other"].copy()

# Filtering to selected MTU window
anc_df_wind = anc_df_wind[anc_df_wind["mtu"].isin(ANC_MTU_WINDOW_WIND)].copy()

# Quick check
display(
    anc_df_wind[["feature", "cluster_group"]]
    .drop_duplicates()
    .sort_values(["cluster_group", "feature"])
    .reset_index(drop=True)
)

In [ ]:
# Step 1: sum contributions per observation and cluster group
wind_grouped = (
    anc_df_wind
    .groupby(
        ["forecast_day", "timestamp", "mtu", "train_days", "cluster_group"],
        as_index=False,
    )["contribution"]
    .sum()
    .rename(columns={"contribution": "group_contribution"})
)

# Step 2: Absolute grouped contribution
wind_grouped["abs_group_contribution"] = wind_grouped["group_contribution"].abs()

# Step 3: ANC per cluster group
wind_anc_summary = (
    wind_grouped
    .groupby(["train_days", "cluster_group"], as_index=False)["abs_group_contribution"]
    .mean()
    .rename(columns={"abs_group_contribution": "ANC"})
    .sort_values("ANC", ascending=False)
    .reset_index(drop=True)
)

display(wind_anc_summary)

In [ ]:
# Export Wind ANC Summary
wind_anc_export = wind_anc_summary.copy()
wind_anc_export["cluster_id"] = (
    wind_anc_export["cluster_group"]
    .str.extract(r"Wind Cluster (\d+)")
    .astype(int)
)

In [ ]:
# Plotting

wind_plot_df = (
    wind_anc_summary.loc[wind_anc_summary["train_days"] == TRAIN_DAYS_ROLLING_ANC]
    .sort_values("ANC", ascending=True)  # for horizontal bar plot
)

plt.figure(figsize=(6, 3))
plt.barh(wind_plot_df["cluster_group"], wind_plot_df["ANC"])
plt.xlabel("ANC")
plt.ylabel("")
plt.title(f"Wind Cluster Importance by ANC (train_days={TRAIN_DAYS_ROLLING_ANC})")
plt.tight_layout()
plt.show()

##### Cluster-Analysis for Solar:

In [ ]:
# Define Feature Groups

def map_solar_feature_to_cluster(feature: str) -> str:
    """
    Map a raw feature name to its solar cluster group.

    Returns the cluster label (e.g. 'Solar Cluster 0') if the feature
    belongs to a solar regressor (ssrd, sw_dir, or sw_dif),
    otherwise returns 'Other'.

    Parameters
    ----------
    feature : str
        Raw feature column name from the LEAR model.

    Returns
    -------
    str
        Solar cluster label (e.g. 'Solar Cluster 0'), or 'Other' if the
        feature does not belong to a solar cluster.
    """
    f = feature.lower()
    if not any(s in f for s in ["ssrd", "sw_dir", "sw_dif"]):
        return "Other"
    match = re.search(r'cluster_(\d+)_h\d+', f)
    if match:
        return f"Solar Cluster {match.group(1)}"
    return "Other"

# Apply mapping – keep only solar features
anc_df_solar = anc_df.copy()
anc_df_solar["cluster_group"] = anc_df_solar["feature"].apply(map_solar_feature_to_cluster)
anc_df_solar = anc_df_solar[anc_df_solar["cluster_group"] != "Other"].copy()

# Filtering to selected MTU window
anc_df_solar = anc_df_solar[anc_df_solar["mtu"].isin(ANC_MTU_WINDOW_SOLAR)].copy()

# Quick check
display(
    anc_df_solar[["feature", "cluster_group"]]
    .drop_duplicates()
    .sort_values(["cluster_group", "feature"])
    .reset_index(drop=True)
)

In [ ]:
# Step 1: sum contributions per observation and cluster group
solar_grouped = (
    anc_df_solar
    .groupby(
        ["forecast_day", "timestamp", "mtu", "train_days", "cluster_group"],
        as_index=False,
    )["contribution"]
    .sum()
    .rename(columns={"contribution": "group_contribution"})
)

# Step 2: Absolute grouped contribution
solar_grouped["abs_group_contribution"] = solar_grouped["group_contribution"].abs()

# Step 3: ANC per cluster group
solar_anc_summary = (
    solar_grouped
    .groupby(["train_days", "cluster_group"], as_index=False)["abs_group_contribution"]
    .mean()
    .rename(columns={"abs_group_contribution": "ANC"})
    .sort_values("ANC", ascending=False)
    .reset_index(drop=True)
)

display(solar_anc_summary)

In [ ]:
# Export Solar ANC Summary
solar_anc_export = solar_anc_summary.copy()
solar_anc_export["cluster_id"] = (
    solar_anc_export["cluster_group"]
    .str.extract(r"Solar Cluster (\d+)")
    .astype(int)
)

In [ ]:
# Plotting
solar_plot_df = (
    solar_anc_summary.loc[solar_anc_summary["train_days"] == TRAIN_DAYS_ROLLING_ANC]
    .sort_values("ANC", ascending=True)
)

plt.figure(figsize=(6, 3))
plt.barh(solar_plot_df["cluster_group"], solar_plot_df["ANC"])
plt.xlabel("ANC")
plt.ylabel("")
plt.title(f"Solar Cluster Importance by ANC (train_days={TRAIN_DAYS_ROLLING_ANC})")
plt.tight_layout()
plt.show()

#### Save results

In [ ]:
EXPORT_PATH_ANC_FEATURES.parent.mkdir(parents=True, exist_ok=True)
anc_summary_df.to_csv(EXPORT_PATH_ANC_FEATURES, index=True)
wind_anc_export.to_csv(EXPORT_PATH_ANC_WIND, index=False)
solar_anc_export.to_csv(EXPORT_PATH_ANC_SOLAR, index=False)
# Save config
anc_config = {
    "experiment_name": EXPERIMENT_NAME_ANC,
    "use_exaa":        USE_EXAA_ANC,
    "use_exaa_only":   USE_EXAA_ONLY_ANC,
    "n_clusters":      None if USE_EXAA_ONLY_ANC else N_CLUSTERS_ANC,
    "weather_source":  None if USE_EXAA_ONLY_ANC else WEATHER_SOURCE_ANC,
    "train_days":      TRAIN_DAYS_ROLLING_ANC,
    "lars_start_date": str(LARS_START_DATE_ANC.date()),
    "test_start":      str(TEST_START_ANC.date()),
    "test_end":        str(TEST_END_ANC.date()),
}
with open(EXPORT_PATH_ANC_BASE / "config.json", "w") as f:
    json.dump(anc_config, f, indent=2)
print(f"✓ Saved: anc_features | anc_wind | anc_solar | config  →  {EXPORT_PATH_ANC_BASE}")